# 🧠 Transformer (GPT-style) from Scratch using NumPy — Deep Dive
This notebook is a **comprehensive theory + implementation guide**.

Each step includes:
- Deep conceptual explanation
- Mathematical formulation
- Why it matters (intuition + significance)
- Clean NumPy implementation

---
By the end, you will understand transformers at a **first-principles level**.


## Step 0: Imports and Setup
We use NumPy to manually build everything.

**Why NumPy?**
- Forces you to understand tensor operations
- No hidden abstractions
- Pure mathematical implementation

In [ ]:
import numpy as np
np.random.seed(42)

## Step 1: Tokenization

### 💡 Concept
Neural networks cannot process text directly. We convert words into integers.

### 📐 Mathematical View
We define a mapping:
$$ V: word \rightarrow integer $$

### 🎯 Significance
- Forms the discrete input space
- Enables embedding lookup


In [ ]:
sentences = [
    "i love ai",
    "i love machine learning",
    "ai is powerful",
    "machine learning is fun",
    "i enjoy learning"
]

vocab = {}
for s in sentences:
    for w in s.split():
        if w not in vocab:
            vocab[w] = len(vocab)

inv_vocab = {i:w for w,i in vocab.items()}

def tokenize(s):
    return [vocab[w] for w in s.split()]

data = [tokenize(s) for s in sentences]
vocab_size = len(vocab)
print(vocab)

## Step 2: Embeddings

### 💡 Concept
Map discrete tokens into continuous vector space.

### 📐 Formula
$$ x_i = E[t_i] $$
Where:
- $E \in \mathbb{R}^{V \times d}$
- $t_i$ is token index

### 🎯 Significance
- Captures semantic similarity
- Enables gradient-based learning


In [ ]:
d_model = 16
embedding_matrix = np.random.randn(vocab_size, d_model)

def embed(tokens):
    return np.array([embedding_matrix[t] for t in tokens])

## Step 3: Positional Encoding

### 💡 Problem
Transformer has no inherent notion of order.

### 📐 Formula
$$PE(pos, 2i) = sin(pos / 10000^{2i/d})$$
$$PE(pos, 2i+1) = cos(pos / 10000^{2i/d})$$

### 🎯 Significance
- Injects position information
- Enables relative distance reasoning


In [ ]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i] = np.sin(pos / (10000 ** (i / d_model)))
            if i+1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE

## Step 4: Scaled Dot-Product Attention

### 💡 Concept
Each token attends to all others.

### 📐 Formula
$$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

### 🧠 Breakdown
- $QK^T$: similarity
- scaling: stability
- softmax: probability distribution

### 🎯 Significance
- Captures relationships between tokens
- Core of transformer


In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def init_weights(d):
    return np.random.randn(d, d)

Wq, Wk, Wv = init_weights(d_model), init_weights(d_model), init_weights(d_model)

def compute_qkv(x):
    return x @ Wq, x @ Wk, x @ Wv

def causal_mask(seq_len):
    return np.tril(np.ones((seq_len, seq_len)))

def attention(Q,K,V,mask):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    scores = np.where(mask==0, -1e9, scores)
    weights = softmax(scores)
    return weights @ V

## Step 5: Multi-Head Attention

### 💡 Concept
Multiple attention heads learn different patterns.

### 📐 Formula
$$MultiHead = Concat(head_1,...,head_h)W_o$$

### 🎯 Significance
- Parallel representation learning
- Richer contextual understanding


In [ ]:
def split_heads(x, h):
    seq_len, d = x.shape
    return x.reshape(seq_len, h, d//h)

def combine_heads(x):
    seq_len, h, d = x.shape
    return x.reshape(seq_len, h*d)

def multi_head(x, h=2):
    Q,K,V = compute_qkv(x)
    mask = causal_mask(x.shape[0])

    Q = split_heads(Q,h)
    K = split_heads(K,h)
    V = split_heads(V,h)

    heads = []
    for i in range(h):
        heads.append(attention(Q[:,i,:],K[:,i,:],V[:,i,:],mask))

    return combine_heads(np.stack(heads,axis=1))

## Step 6: Feed Forward + LayerNorm

### 📐 Formula
$$FFN(x)=max(0,xW_1)W_2$$

### 🎯 Significance
- Introduces non-linearity
- Improves representation capacity


In [ ]:
def layer_norm(x):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x-mean)/(std+1e-6)

W1 = np.random.randn(d_model,d_model*2)
W2 = np.random.randn(d_model*2,d_model)

def relu(x):
    return np.maximum(0,x)

def ffn(x):
    return relu(x@W1)@W2

## Step 7: GPT Block

### Structure
Masked Attention → Add&Norm → FFN → Add&Norm

### 🎯 Significance
- Residual connections stabilize training
- Deep stacking possible


In [ ]:
def gpt_block(x):
    x = layer_norm(x + multi_head(x))
    x = layer_norm(x + ffn(x))
    return x

## Step 8: GPT Model
Stack multiple blocks

In [ ]:
def gpt_model(x, layers=2):
    for _ in range(layers):
        x = gpt_block(x)
    return x

W_out = np.random.randn(d_model,vocab_size)

def gpt_forward(tokens):
    x = embed(tokens)
    x += positional_encoding(len(tokens),d_model)
    x = gpt_model(x)
    logits = x@W_out
    return softmax(logits)

## Step 9: Training Objective

### 📐 Loss
$$L = -log(p_{correct})$$

### 🎯 Significance
- Encourages correct predictions
- Drives parameter updates


In [ ]:
def loss_fn(probs,target):
    return -np.log(probs[target]+1e-9)

## Step 10: Text Generation

Iteratively predict next token

In [ ]:
def generate(start,max_len=5):
    tokens = tokenize(start)
    for _ in range(max_len):
        probs = gpt_forward(tokens)
        tokens.append(np.argmax(probs[-1]))
    return " ".join([inv_vocab[t] for t in tokens])

print(generate("i"))